# Imports

In [ ]:
import os
import utils
import xml.etree.ElementTree as ET
import numpy as np
import matplotlib.pyplot as plt
import ceinms
import openSim
import pandas as pd
import shutil
import opensim as osim

import settings
# import muscleOptimizer


# Export session c3d

In [ ]:
subject = 'HC835B' 
session = 'Session1' # '25_03_31' '22_07_06' walking 09-03-2026
emg_string_list = ['1_GLUT_MED_1(3)', '2_BICEP_FEM_1(3)' , 'RECT_FEM_1(3)' , '4_TIBIAL_ANT_1(3)']

files_session = os.listdir(os.path.join(utils.SIMULATIONS_DIR, subject, session))

for folder in files_session:
    folder_path = os.path.join(utils.SIMULATIONS_DIR, subject, session, folder)
    if os.path.isdir(folder_path):
        for file in os.listdir(folder_path):
            if file.endswith('.c3d'):
                openSim.export_c3d(c3d_file_path=os.path.join(folder_path, file), emg_string_list=emg_string_list)

In [ ]:
model_path = r'C:\Git\powerlifing_model_clean\models\tps\final_results\personalized\tps_transformed.osim'

def reset_osim_model(model_path):
    '''Read XML tree and saave it back to reset any internal state in the model file.'''
    tree = ET.parse(model_path)

    utils.save_pretty_xml(tree, model_path)  # Save back to the original file

reset_osim_model(model_path)

# Single trial analysis

## Initiate trial

In [ ]:
subject = '019'
trial = 'SJ_pre3'  

trialPath = os.path.join(utils.SIMULATIONS_DIR, subject, trial) # Walking_02, Squat_bw_01, Squat_35kg_01
print(f'Analyzing trial at path: {trialPath}')

analysis = utils.Analyse(trialPath)
analysis._reset_settings_xml()

print("Main inputs:")
print(f"Model file: {os.path.join(analysis.path, analysis.model_dir)}")
print(f"time range: {analysis.time_range}")
print(f"Body mass: {analysis.body_mass} kg")
print(f"Body mass grf: {analysis.get_body_mass_from_grf(update=True)} kg")
print(f"Body mass osim: {analysis.get_body_mass()} kg")

analysis.update_trial_attribute('Target_Muscles', 'all')
analysis.delete_trial_attribute('EMG_muscle_mapping')
analysis.delete_trial_attribute('Objective_Functions')


# open settings.xml in the editor
settings_xml_path = os.path.join(trialPath, 'trial_settings.xml')
os.startfile(settings_xml_path)

## reload settings

In [ ]:
analysis.load_settings(analysis.settingsXML)
print(f'{analysis.model_dir}')

## Reset settings

In [ ]:
analysis._reset_settings_xml()

## Export C3D

In [ ]:
files_session = os.listdir(os.path.join(utils.SIMULATIONS_DIR, subject))

for file in files_session:
    if file.endswith('.c3d'):
        openSim.export_c3d(c3d_file_path=os.path.join(utils.SIMULATIONS_DIR, subject, file))

## Get events

In [ ]:
def detect_trc_events_from_marker_y(trc_path, marker_name="USACR", output_csv=None):
    # Read TRC header rows
    with open(trc_path, "r") as f:
        lines = f.readlines()

    header_markers = lines[3].rstrip("\n").split("\t")
    header_coords = lines[4].rstrip("\n").split("\t")

    # Build column names (e.g., USACR_X, USACR_Y, USACR_Z)
    columns = []
    current_marker = ""
    ncols = max(len(header_markers), len(header_coords))

    for i in range(ncols):
        h1 = header_markers[i].strip() if i < len(header_markers) else ""
        h2 = header_coords[i].strip() if i < len(header_coords) else ""

        if h1:
            current_marker = h1

        if h1 in ("Frame#", "Time"):
            columns.append(h1)
        elif current_marker and h2:
            columns.append(f"{current_marker}_{h2[0].upper()}")
        else:
            columns.append(f"col_{i}")

    # Load data rows
    df = pd.read_csv(
        trc_path,
        sep="\t",
        skiprows=5,
        header=None,
        names=columns,
        engine="python"
    ).dropna(axis=1, how="all")

    df["Time"] = pd.to_numeric(df["Time"], errors="coerce")

    y_col = f"{marker_name}_Y"
    if y_col not in df.columns:
        raise ValueError(f"Marker column not found: {y_col}")

    y = pd.to_numeric(df[y_col], errors="coerce")
    valid = df["Time"].notna() & y.notna()
    df_valid = df.loc[valid].copy()
    y_valid = y.loc[valid]

    start_idx = y_valid.idxmin()
    end_idx = y_valid.idxmax()

    start_time = float(df_valid.loc[start_idx, "Time"])
    end_time = float(df_valid.loc[end_idx, "Time"])

    events = pd.DataFrame({
        "event": ["start", "end"],
        "time": [start_time, end_time],
        "marker": [marker_name, marker_name],
        "y_value": [float(y_valid.loc[start_idx]), float(y_valid.loc[end_idx])],
    })

    if output_csv is None:
        output_csv = os.path.join(os.path.dirname(trc_path), "events.csv")
    events.to_csv(output_csv, index=False)

    return (start_time, end_time), events, output_csv


trc_path = os.path.join(trialPath, "markers_experimental.trc")
time_range, events_df, events_file = detect_trc_events_from_marker_y(trc_path, marker_name="USACR")

print("Detected time range:", time_range)
print("Saved events to:", events_file)
events_df

analysis._reset_settings_xml()


## Increase model isometric force

In [ ]:
analysis.increase_muscle_force(factor=3.00)

## Run Inverse Kinematics

In [ ]:
analysis.update_trial_attribute('replace', True)
analysis.run_ik()

## Run Inverse Dynamics

In [ ]:
analysis.update_trial_attribute('replace', True)
analysis.run_id()

## Run Muscle Analysis

In [ ]:
analysis.update_trial_attribute('replace', True)
analysis.run_ma()

## Run Static Optimisation

In [ ]:
analysis.update_trial_attribute('replace', True)

if not analysis.model_name.__contains__('_increased_3.00'):
    new_model_name = analysis.model_name.replace('.osim', '_increased_3.00.osim')
    analysis.update_model(new_model_name)

In [ ]:
analysis.load_settings(analysis.settingsXML)
analysis.run_so()

In [ ]:
analysis.plot_so()

## Calculate muscle moments SO

In [ ]:
analysis.calculate_muscle_moments(forces_type='so')

## Run Joint Reaction Analysis (with SO results)

In [ ]:
analysis.update_trial_attribute('replace', True)
analysis.run_jra()

## Create ceinms setup files

In [ ]:
if analysis.model_name.__contains__('_increased_3.00'):
    new_model_name = analysis.model_name.replace('_increased_3.00.osim', '.osim')
    analysis.update_model(new_model_name)

In [ ]:
analysis.update_trial_attribute('replace', True)
analysis.create_ceinms_input_data()

In [ ]:
analysis.update_trial_attribute('replace', True)

analysis.create_ceinms_model()


In [ ]:
analysis.create_ceinms_calibration_cfg(calibration_trial_names=settings.calibration_trials)

analysis.create_ceinms_calibration_setup()

In [ ]:
analysis.update_trial_attribute('replace', 'True')
analysis.create_excitation_generator()

## ceinms calibration

In [ ]:
analysis.run_ceinms_calibration()


## ceinms exe setups

In [ ]:
# print ceinms settings
print('CEIMS Settings:')
print(f'alpha: {analysis.alpha}')
print(f'beta: {analysis.beta}')
print(f'gamma: {analysis.gamma}')
print(f'ceinms_exe_dir: {analysis.ceinms_exe_dir}')

analysis.update_trial_attribute('jra_forces_ceinms', f'Execution_a{analysis.alpha}_b{analysis.beta}_g{analysis.gamma}/MuscleForces.sto')

In [ ]:
analysis.create_ceinms_exe_cfg()

analysis.create_ceinms_exe_setup()

In [ ]:
analysis.create_ceinms_cfg_from_excitation_generator()

## run ceinms exe

In [ ]:
analysis.run_ceinms_exe()


## JRA ceinms 

In [ ]:
# Use if need to replace model for CEINMS execution 
if analysis.model_name.__contains__('_increased_3.00'):
    new_model_name = analysis.model_name.replace('_increased_3.00.osim', '.osim')
    analysis.update_model(new_model_name)

In [ ]:
analysis.update_trial_attribute('replace', True)
analysis.run_jra_ceinms()

## Compare muscle moments 

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

ik_columns = settings.DOFs
id_columns = [f'{dof}_moment' for dof in ik_columns]

id = utils.load_any_data_file(os.path.join(analysis.path, analysis.id))
so_forces = utils.load_any_data_file(os.path.join(analysis.path, analysis.so_forces))
ceinms_forces = utils.load_any_data_file(os.path.join(analysis.path, analysis.jra_forces_ceinms))

fig, ax = plt.subplots(nrows=len(ik_columns), ncols=2, figsize=(28, 16))
fontsize = 25

fig_int = make_subplots(rows=len(ik_columns), cols=2, shared_xaxes=True,
                        subplot_titles=[f"{dof} - SO" if i % 2 == 0 else f"{dof} - CEINMS"
                                        for dof in ik_columns for i in range(2)])

for count, dof in enumerate(ik_columns):
    ma = utils.load_any_data_file(os.path.join(analysis.path, analysis.ma, f'_MuscleAnalysis_MomentArm_{dof}.sto'))

    muscle_list = [col for col in so_forces.columns if col != 'time']
    muscles = openSim.find_non_zero_mom_arm_muscles(ma, muscle_list)
    print(f"Non-zero moment arm muscles for {dof}: {muscles}")

    colors = plt.cm.tab20(np.linspace(0, 1, max(len(muscles), 1)))

    if count == 0:
        ax[count, 0].set_title('SO', fontsize=fontsize)
        ax[count, 1].set_title('CEINMS', fontsize=fontsize)

    # Plot ID (static)
    ax[count, 0].plot(id['time'], id[f'{dof}_moment'], label='ID', color='blue')
    ax[count, 0].set_ylabel(f'{dof} (Nm)', fontsize=fontsize)
    ax[count, 0].tick_params(labelsize=fontsize)

    ax[count, 1].plot(id['time'], id[f'{dof}_moment'], label='ID', color='blue')
    ax[count, 1].tick_params(labelsize=fontsize)

    
    # Plot muscle moments SO (static)
    for i, m in enumerate(muscles):
        ax[count, 0].plot(so_forces['time'], so_forces[m] * ma[m],
                          label=m, color=colors[i], linestyle='--')

    sum_moments_so = so_forces[muscles].mul(ma[muscles], axis=0).sum(axis=1)
    ax[count, 0].fill_between(so_forces['time'], 0, sum_moments_so, color='grey', alpha=0.2, label='Sum')

    # Plot muscle moments CEINMS (static)
    for i, m in enumerate(muscles):
        ax[count, 1].plot(ceinms_forces['time'], ceinms_forces[m] * ma[m],
                          label=m, color=colors[i], linestyle='--')
    
    sum_moments_ce = ceinms_forces[muscles].mul(ma[muscles], axis=0).sum(axis=1)
    ax[count, 1].fill_between(ceinms_forces['time'], 0, sum_moments_ce, color='grey', alpha=0.2, label='Sum')

    # add RMSE and R2 values between ID and SO, and ID and CEINMS
    rmse_so = utils.rmse(id[f'{dof}_moment'], sum_moments_so)
    r2_so = utils.rsquared(id[f'{dof}_moment'], sum_moments_so)
    rmse_ce = utils.rmse(id[f'{dof}_moment'], sum_moments_ce)
    r2_ce = utils.rsquared(id[f'{dof}_moment'], sum_moments_ce)
    textstr_so = f'RMSE={rmse_so:.2f} Nm\nR²={r2_so:.3f}'
    textstr_ce = f'RMSE={rmse_ce:.2f} Nm\nR²={r2_ce:.3f}'
    ax[count, 0].text(0.95, 0.95, textstr_so, transform=ax[count, 0].transAxes, fontsize=fontsize,
                      verticalalignment='top', horizontalalignment='right',
                      bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    ax[count, 1].text(0.95, 0.95, textstr_ce, transform=ax[count, 1].transAxes, fontsize=fontsize,
                      verticalalignment='top', horizontalalignment='right',
                      bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))    


fig.tight_layout()
save_path = os.path.join(analysis.path, 'muscle_moments.png')
fig.savefig(save_path)
print(f"Muscle moments comparison plot saved: {save_path}")



In [ ]:
utils.convert_to_interactive_fig(fig, html_path=os.path.join(analysis.path, 'muscle_moments_interactive.html'))

## Compare JRA SO and CEINMS

In [ ]:
jra_dict = {'hip': ['hip_r_on_femur_r_in_femur_r_fx', 'hip_r_on_femur_r_in_femur_r_fy', 'hip_r_on_femur_r_in_femur_r_fz'],
            'knee': ['knee_r_on_tibia_r_in_tibia_r_fx', 'knee_r_on_tibia_r_in_tibia_r_fy', 'knee_r_on_tibia_r_in_tibia_r_fz'],
            'ankle': ['ankle_r_on_talus_r_in_talus_r_fx', 'ankle_r_on_talus_r_in_talus_r_fy', 'ankle_r_on_talus_r_in_talus_r_fz']}


jra_dict = {'hip': ['hip_r_on_femur_r_in_femur_r_fx', 'hip_r_on_femur_r_in_femur_r_fy', 'hip_r_on_femur_r_in_femur_r_fz'],
            'knee': ['med_cond_joint_r_on_med_cond_r_in_med_cond_r_fx', 'med_cond_joint_r_on_med_cond_r_in_med_cond_r_fy', 'med_cond_joint_r_on_med_cond_r_in_med_cond_r_fz'],
            'ankle': ['ankle_r_on_talus_r_in_talus_r_fx', 'ankle_r_on_talus_r_in_talus_r_fy', 'ankle_r_on_talus_r_in_talus_r_fz']}

In [ ]:
jra_Scaled_so = utils.load_any_data_file(os.path.join(analysis.path, analysis.jra))
jra_Scaled_ceinms = utils.load_any_data_file(os.path.join(analysis.path, analysis.jra_ceinms))

fig, ax = plt.subplots(3,   4, figsize=(15, 8))

labels = ['SO', 'CEINMS']
lineStyles = ['-', '--']
count = -1
for df in [jra_Scaled_so, jra_Scaled_ceinms]:
    count += 1
    
    # plot Hip first row    
    x = df[jra_dict['hip'][0]]
    y = df[jra_dict['hip'][1]]
    z = df[jra_dict['hip'][2]]
    sum = np.linalg.norm([x,y,z], axis=0)
    ax[0, 0].plot(df['time'], x, label=labels[count], linestyle=lineStyles[count])
    ax[0, 1].plot(df['time'], y, label=labels[count], linestyle=lineStyles[count])
    ax[0, 2].plot(df['time'], z, label=labels[count], linestyle=lineStyles[count])
    ax[0, 3].plot(df['time'], sum, label=labels[count], linestyle=lineStyles[count])
    
    ax[0, 0].set_ylabel(' Hip Force (N)')
    ax[0, 0].set_title('X (+ anterior, - posterior)')
    ax[0, 1].set_title('Y (+ medial, - lateral)')
    ax[0, 2].set_title('Z (+ superior, - inferior)')
    ax[0, 3].set_title('Resultant Force Magnitude')

    # plot Knee second row
    x = df[jra_dict['knee'][0]]
    y = df[jra_dict['knee'][1]]
    z = df[jra_dict['knee'][2]]

    sum = np.linalg.norm([x,y,z], axis=0)
    ax[1, 0].plot(df['time'], x, label=labels[count], linestyle=lineStyles[count])
    ax[1, 1].plot(df['time'], y, label=labels[count], linestyle=lineStyles[count])
    ax[1, 2].plot(df['time'], z, label=labels[count], linestyle=lineStyles[count])
    ax[1, 3].plot(df['time'], sum, label=labels[count], linestyle=lineStyles[count])

    ax[1, 0].set_ylabel(' Knee Force (N)')
    
    
    # plot Ankle last row
    x = df[jra_dict['ankle'][0]]
    y = df[jra_dict['ankle'][1]]
    z = df[jra_dict['ankle'][2]]
    sum = np.linalg.norm([x,y,z], axis=0)
    ax[2, 0].plot(df['time'], x, label=labels[count], linestyle=lineStyles[count])
    ax[2, 1].plot(df['time'], y, label=labels[count], linestyle=lineStyles[count])
    ax[2, 2].plot(df['time'], z, label=labels[count], linestyle=lineStyles[count])
    ax[2, 3].plot(df['time'], sum, label=labels[count], linestyle=lineStyles[count])

    ax[2, 0].set_ylabel(' Ankle Force (N)')
    
        

ax[0, 0].legend()
plt.tight_layout()
save_path = os.path.join(analysis.path, f'jra_so_vs_ceinms.png')
plt.savefig(save_path)
print(f"JRA comparison plot saved: {save_path}")
    



## Summary plot trial

In [ ]:
analysis.plot_summary()

## push to git (trial)

In [ ]:
analysis.push_trial_results_to_git()

## push to git (subject)

In [ ]:
analysis.push_subject_results_to_git()

# Troubleshoot

In [ ]:
analysis.load_settings(analysis.settingsXML)
osim_modelPath = analysis.model_dir
ik_output = os.path.join(analysis.path, analysis.ik)
grf_xml = os.path.join(analysis.path, analysis.setup_grf)
setup_jra_xml = os.path.join(analysis.path, analysis.setup_jra)
actuators_xml = None
muscle_force_path = os.path.join(analysis.path, analysis.jra_forces_ceinms).replace('MuscleForces', 'MuscleForces_updated')
saveFileName = os.path.join(analysis.path, "JRA_results.sto")

openSim.run_jra(osim_modelPath=osim_modelPath, 
                ik_output=ik_output, 
                grf_xml=grf_xml, 
                setup_xml=setup_jra_xml, 
                actuators=actuators_xml, 
                muscle_force_path=muscle_force_path, 
                saveFileName=saveFileName)

In [ ]:
setupXML_path = r'C:\Git\powerlifing_model\simulations\Athlete_03\25_03_31\Walking_02\ceinms_setup.xml'
ceinms.executable(setupXML_path)

## Compare multiple files

In [ ]:
filepath1 = os.path.join(analysis.path, analysis.jra)
filepath2 = os.path.join(analysis.path, analysis.jra_ceinms)
filepath3 = os.path.join(analysis.path, analysis.jra_ceinms).replace('MuscleForces', 'MuscleForces_updated')

files = {'so': utils.load_any_data_file(filepath1),
         'ceinms': utils.load_any_data_file(filepath2),
         'ceinms_with_SO_updated': utils.load_any_data_file(filepath3)}

results_dir = os.path.join(utils.RESULTS_DIR, 'compare')
os.makedirs(results_dir, exist_ok=True)

common_columns = ['hip_r_on_femur_r_in_femur_r_fx', 'hip_r_on_femur_r_in_femur_r_fy', 'hip_r_on_femur_r_in_femur_r_fz', 'walker_knee_r_on_tibia_r_in_tibia_r_fx', 'walker_knee_r_on_tibia_r_in_tibia_r_fy', 'walker_knee_r_on_tibia_r_in_tibia_r_fz', 'ankle_r_on_talus_r_in_talus_r_fx', 'ankle_r_on_talus_r_in_talus_r_fy', 'ankle_r_on_talus_r_in_talus_r_fz']

nrows, ncols = utils.calculate_nRows_nCols(len(common_columns))

fig, ax = plt.subplots(nrows=nrows, ncols=ncols, figsize=(10, 6*nrows))

for file_key, data in files.items():
    print(f"{file_key} columns: {data.columns}")

    for idx, column in enumerate(common_columns):
        row = idx // ncols
        col_name = idx % ncols
        ax[row, col_name].plot(data['time'], data[column], label=file_key)
        ax[row, col_name].set_title(column)
        ax[row, col_name].set_xlabel('Time (s)')
        ax[row, col_name].set_ylabel('Value')
        ax[row, col_name].legend()
# plt.tight_layout()
save_path = os.path.join(results_dir, 'comparison_plot.png')

In [ ]:
# compare columns SO vs CEINMS forces
ceinms_forces = utils.load_any_data_file(os.path.join(analysis.path, analysis.jra_forces_ceinms))
so_forces = utils.load_any_data_file(os.path.join(analysis.path, analysis.so_forces))

for column in so_forces.columns:
    if not column in ceinms_forces.columns:
        print(f"Column {column} not found in CEINMS forces.")
        
        # add column to ceinms_forces with the same valuse as SO
        ceinms_forces[column] = so_forces[column]

utils.write_sto_file(ceinms_forces, os.path.join(analysis.path, analysis.jra_forces_ceinms).replace('.sto', '_updated.sto'))
print(f"Saved updated CEINMS forces to {os.path.join(analysis.path, analysis.jra_forces_ceinms).replace('.sto', '_updated.sto')}")
        

# Compare models and tasks


## Initiate settings

### --- settings Catei, Lernagopal, GPK, Ulrich

In [ ]:

subject_names = {'Scaled (Cateli)': 'Athlete_03',
                'Scaled (Lernagopal)': 'Athlete_03_Lernagopal',
                'Scaled (Lernagopal Optimised)': 'Athlete_03_Lernagopal_optimised',
                'Scaled (GPK)': 'Athlete_03_GPK',
                'Scaled (GPK) - CEINMS': 'Athlete_03_GPK',
                'Katya MRI (Cateli)': 'Athlete_03_MRI_Katya',
                'Scaled (Uhlrich)': 'Athlete_03_Uhlrich'}

session = '25_03_31'
trialName = 'Squat_bw_01' # 'Walking_02', 'Squat_bw_01', 'Squat_35kg_01'


ik_columns = ['hip_flexion_r', 'hip_adduction_r', 'hip_rotation_r', 'knee_angle_r','knee_adduction_r', 'ankle_angle_r']
id_columns = ['hip_flexion_r_moment', 'hip_adduction_r_moment', 'hip_rotation_r_moment', 'knee_angle_r_moment', 'knee_adduction_r_moment','ankle_angle_r_moment']

muscleGroups = {'R Gluteus maximus':['glmax1_r','glmax2_r','glmax3_r'],
                        'R Gluteus medius':['glmed1_r','glmed2_r','glmed3_r'],
                        'R Gluteus minimus':['glmin1_r','glmin2_r','glmin3_r'], 
                        'R Adductor Magnus': ['addmagDist_r','addmagIsch_r','addmagMid_r','addmagProx_r'],
                        'R Biceps Femoris': ['bflh_r','bfsh_r'],
                        'R Semimembranosus': ['semimem_r'],
                        'R Semitendinosus': ['semiten_r'],
                        'R Rectus Femoris': ['recfem_r'],
                        'R Vasti':['vasint_r','vaslat_r','vasmed_r'],
                        'R Triceps Surae': ['soleus_r','gaslat_r','gasmed_r'],
                        'L Gluteus maximus':['glmax1_l','glmax2_l','glmax3_l'],
                        'L Gluteus medius':['glmed1_l','glmed2_l','glmed3_l'],
                        'L Gluteus minimus':['glmin1_l','glmin2_l','glmin3_l'], 
                        'L Adductor Magnus': ['addmagDist_l','addmagIsch_l','addmagMid_l','addmagProx_l'],
                        'L Biceps Femoris': ['bflh_l','bfsh_l'],
                        'L Semimembranosus': ['semimem_l'],
                        'L Semitendinosus': ['semiten_l'],
                        'L Rectus Femoris': ['recfem_l'],
                        'L Vasti':['vasint_l','vaslat_l','vasmed_l'],
                        'L Triceps Surae': ['soleus_l','gaslat_l','gasmed_l'],
                        }

colors = {'Scaled (Cateli)': 'green',
          'Scaled (Lernagopal)': 'blue',
          'Scaled (Lernagopal Optimised)': 'red',
          'Scaled (GPK)': 'purple',
          'Scaled (GPK) - CEINMS': 'cyan',
          'Katya MRI (Cateli)': 'orange',
          'Scaled (Uhlrich)': 'brown'}

forces_type = {'Scaled (Cateli)': 'SO',
               'Scaled (Lernagopal)': 'SO',
               'Scaled (Lernagopal Optimised)': 'SO',
               'Scaled (GPK)': 'SO',
               'Scaled (GPK) - CEINMS': 'CEINMS',
                'Katya MRI (Cateli)': 'SO',
               'Scaled (Uhlrich)': 'SO'}

lineStyles = {'Scaled (Cateli)': '-',
              'Scaled (Lernagopal)': '--',
              'Scaled (Lernagopal Optimised)': '-.',
              'Scaled (GPK)': ':',
              'Scaled (GPK) - CEINMS': '-',
              'Katya MRI (Cateli)': ':',
              'Scaled (Uhlrich)':'--'}

labels = colors.keys()

# results_dir = os.path.join(utils.RESULTS_DIR, 'Katya_vs_BG')
results_dir = os.path.join(utils.RESULTS_DIR, 'compare_tasks_lernagopal_optimised')
os.makedirs(results_dir, exist_ok=True)

trials = {}
for label, subject in zip(labels, subject_names.values()):
    trialPath = os.path.join(utils.SIMULATIONS_DIR, subject, session, trialName)
    trials[label] = utils.Analyse(trialPath)

print(f'Results to be saved to: {results_dir}')

### --- settings GPK MRI

#### Walking_02

In [ ]:
model_config = {
    'Scaled (Cateli)':        {'subject': 'Athlete_03',          'color': 'green',  'force_type': 'SO',     'line_style': '-'},
    'Scaled (Cateli) - CEINMS':        {'subject': 'Athlete_03',          'color': 'green',  'force_type': 'CEINMS',     'line_style': '--'},
    'Scaled (Lernagopal)':    {'subject': 'Athlete_03_Lernagopal','color': 'blue',   'force_type': 'SO',     'line_style': '-'},
    'Scaled (Lernagopal) - CEINMS':    {'subject': 'Athlete_03_Lernagopal','color': 'blue',   'force_type': 'CEINMS',     'line_style': '--'},
    'Scaled (GPK)':           {'subject': 'Athlete_03_GPK',      'color': 'red',    'force_type': 'SO',     'line_style': '-.'},
    'Scaled (GPK) - CEINMS': {'subject': 'Athlete_03_GPK',      'color': 'red',    'force_type': 'CEINMS', 'line_style': '--'},
    'MRI (GPK)':       {'subject': 'Athlete_03_GPK_MRI',  'color': 'purple', 'force_type': 'SO',     'line_style': '-'},
    'MRI (GPK) - CEINMS':  {'subject': 'Athlete_03_GPK_MRI',  'color': 'magenta','force_type': 'CEINMS', 'line_style': '--'},
}

subject_names = {k: v['subject'] for k, v in model_config.items()}
colors = {k: v['color'] for k, v in model_config.items()}
forces_type = {k: v['force_type'] for k, v in model_config.items()}
lineStyles = {k: v['line_style'] for k, v in model_config.items()}
labels = list(model_config.keys())


ik_columns = ['hip_flexion_r', 'hip_adduction_r', 'hip_rotation_r', 'knee_angle_r','knee_adduction_r', 'ankle_angle_r']
id_columns = ['hip_flexion_r_moment', 'hip_adduction_r_moment', 'hip_rotation_r_moment', 'knee_angle_r_moment', 'knee_adduction_r_moment','ankle_angle_r_moment']

muscleGroups = {'R Gluteus maximus':['glmax1_r','glmax2_r','glmax3_r'],
                        'R Gluteus medius':['glmed1_r','glmed2_r','glmed3_r'],
                        'R Gluteus minimus':['glmin1_r','glmin2_r','glmin3_r'], 
                        'R Adductor Magnus': ['addmagDist_r','addmagIsch_r','addmagMid_r','addmagProx_r'],
                        'R Biceps Femoris': ['bflh_r','bfsh_r'],
                        'R Semimembranosus': ['semimem_r'],
                        'R Semitendinosus': ['semiten_r'],
                        'R Rectus Femoris': ['recfem_r'],
                        'R Vasti':['vasint_r','vaslat_r','vasmed_r'],
                        'R Triceps Surae': ['soleus_r','gaslat_r','gasmed_r'],
                        'L Gluteus maximus':['glmax1_l','glmax2_l','glmax3_l'],
                        'L Gluteus medius':['glmed1_l','glmed2_l','glmed3_l'],
                        'L Gluteus minimus':['glmin1_l','glmin2_l','glmin3_l'], 
                        'L Adductor Magnus': ['addmagDist_l','addmagIsch_l','addmagMid_l','addmagProx_l'],
                        'L Biceps Femoris': ['bflh_l','bfsh_l'],
                        'L Semimembranosus': ['semimem_l'],
                        'L Semitendinosus': ['semiten_l'],
                        'L Rectus Femoris': ['recfem_l'],
                        'L Vasti':['vasint_l','vaslat_l','vasmed_l'],
                        'L Triceps Surae': ['soleus_l','gaslat_l','gasmed_l'],
                        }

# results_dir = os.path.join(utils.RESULTS_DIR, 'Katya_vs_BG')
results_dir = os.path.join(utils.RESULTS_DIR, 'GPK_validation')
os.makedirs(results_dir, exist_ok=True)


session = '25_03_31'
trialName = 'Walking_02' # 'Walking_02', 'Squat_bw_01', 'Squat_35kg_01'

trials = {}
for label, subject in zip(labels, subject_names.values()):
    trialPath = os.path.join(utils.SIMULATIONS_DIR, subject, session, trialName)
    trials[label] = utils.Analyse(trialPath)

print(f'Results to be saved to: {results_dir}')


#### Squat_BW_01

In [ ]:
model_config = {
    'Scaled (Cateli)':        {'subject': 'Athlete_03',          'color': 'green',  'force_type': 'SO',     'line_style': '-'},
    'Scaled (Lernagopal)':    {'subject': 'Athlete_03_Lernagopal','color': 'blue',   'force_type': 'SO',     'line_style': '-'},
    'Scaled (GPK)':           {'subject': 'Athlete_03_GPK',      'color': 'red',    'force_type': 'SO',     'line_style': '-.'},
    'Scaled (GPK) - CEINMS': {'subject': 'Athlete_03_GPK',      'color': 'red',    'force_type': 'CEINMS', 'line_style': '--'},
    'MRI (GPK)':       {'subject': 'Athlete_03_GPK_MRI',  'color': 'purple', 'force_type': 'SO',     'line_style': '-'},
    'MRI (GPK) - CEINMS':  {'subject': 'Athlete_03_GPK_MRI',  'color': 'magenta','force_type': 'CEINMS', 'line_style': '--'},
    'MRI_ increased (GPK) - CEINMS':  {'subject': 'Athlete_03_GPK_MRI',  'color': 'magenta','force_type': 'CEINMS', 'line_style': '--'},
    
}

subject_names = {k: v['subject'] for k, v in model_config.items()}
colors = {k: v['color'] for k, v in model_config.items()}
forces_type = {k: v['force_type'] for k, v in model_config.items()}
lineStyles = {k: v['line_style'] for k, v in model_config.items()}
labels = list(model_config.keys())


ik_columns = settings.DOFs
id_columns = settings.DOFs_moments

muscleGroups = {'R Gluteus maximus':['glmax1_r','glmax2_r','glmax3_r'],
                        'R Gluteus medius':['glmed1_r','glmed2_r','glmed3_r'],
                        'R Gluteus minimus':['glmin1_r','glmin2_r','glmin3_r'], 
                        'R Adductor Magnus': ['addmagDist_r','addmagIsch_r','addmagMid_r','addmagProx_r'],
                        'R Biceps Femoris': ['bflh_r','bfsh_r'],
                        'R Semimembranosus': ['semimem_r'],
                        'R Semitendinosus': ['semiten_r'],
                        'R Rectus Femoris': ['recfem_r'],
                        'R Vasti':['vasint_r','vaslat_r','vasmed_r'],
                        'R Triceps Surae': ['soleus_r','gaslat_r','gasmed_r'],
                        'L Gluteus maximus':['glmax1_l','glmax2_l','glmax3_l'],
                        'L Gluteus medius':['glmed1_l','glmed2_l','glmed3_l'],
                        'L Gluteus minimus':['glmin1_l','glmin2_l','glmin3_l'], 
                        'L Adductor Magnus': ['addmagDist_l','addmagIsch_l','addmagMid_l','addmagProx_l'],
                        'L Biceps Femoris': ['bflh_l','bfsh_l'],
                        'L Semimembranosus': ['semimem_l'],
                        'L Semitendinosus': ['semiten_l'],
                        'L Rectus Femoris': ['recfem_l'],
                        'L Vasti':['vasint_l','vaslat_l','vasmed_l'],
                        'L Triceps Surae': ['soleus_l','gaslat_l','gasmed_l'],
                        }

# results_dir = os.path.join(utils.RESULTS_DIR, 'Katya_vs_BG')
results_dir = os.path.join(utils.RESULTS_DIR, 'GPK_validation')
os.makedirs(results_dir, exist_ok=True)
session = '25_03_31'
trialName = 'Squat_BW_01' # 'Walking_02', 'Squat_BW_01', 'Squat_35kg_01'

trials = {}
for label, subject in zip(labels, subject_names.values()):
    trialPath = os.path.join(utils.SIMULATIONS_DIR, subject, session, trialName)
    trials[label] = utils.Analyse(trialPath)

print(f'Results to be saved to: {results_dir}')

#### Squat_35kg_01

In [ ]:
model_config = {
    'Scaled (Cateli)':        {'subject': 'Athlete_03',          'color': 'green',  'force_type': 'SO',     'line_style': '-'},
    'Scaled (Cateli) - CEINMS':        {'subject': 'Athlete_03',          'color': 'green',  'force_type': 'CEINMS',     'line_style': '--'},
    'Scaled (Lernagopal)':    {'subject': 'Athlete_03_Lernagopal','color': 'blue',   'force_type': 'SO',     'line_style': '-'},
    'Scaled (Lernagopal) - CEINMS':    {'subject': 'Athlete_03_Lernagopal','color': 'blue',   'force_type': 'CEINMS',     'line_style': '--'},
    'Scaled (GPK)':           {'subject': 'Athlete_03_GPK',      'color': 'red',    'force_type': 'SO',     'line_style': '-.'},
    'Scaled (GPK) - CEINMS': {'subject': 'Athlete_03_GPK',      'color': 'red',    'force_type': 'CEINMS', 'line_style': '--'},
    'MRI (GPK)':       {'subject': 'Athlete_03_GPK_MRI',  'color': 'purple', 'force_type': 'SO',     'line_style': '-'},
    'MRI (GPK) - CEINMS':  {'subject': 'Athlete_03_GPK_MRI',  'color': 'magenta','force_type': 'CEINMS', 'line_style': '--'},
}

subject_names = {k: v['subject'] for k, v in model_config.items()}
colors = {k: v['color'] for k, v in model_config.items()}
forces_type = {k: v['force_type'] for k, v in model_config.items()}
lineStyles = {k: v['line_style'] for k, v in model_config.items()}
labels = list(model_config.keys())


ik_columns = settings.DOFs
id_columns = settings.DOFs_moments
muscleGroups = settings.Muscle_Groups

# results_dir = os.path.join(utils.RESULTS_DIR, 'Katya_vs_BG')
results_dir = os.path.join(utils.RESULTS_DIR, 'GPK_validation')
os.makedirs(results_dir, exist_ok=True)
session = '25_03_31'
trialName = 'Squat_35kg_01' # 'Walking_02', 'Squat_BW_01', 'Squat_35kg_01'

trials = {}
for label, subject in zip(labels, subject_names.values()):
    trialPath = os.path.join(utils.SIMULATIONS_DIR, subject, session, trialName)
    trials[label] = utils.Analyse(trialPath)

print(f'Results to be saved to: {results_dir}')

## Compare model weights

In [ ]:
# Compare model weights (total model mass) across all loaded trials
model_weights = {}

for model_name, trial in trials.items():

    if not isinstance(trial, utils.Analyse):
        print(f"Skipping {model_name}: not an Analyse instance")
        continue

    try:
        model_path = trial.model_dir
        if not os.path.isabs(model_path):
            model_path = os.path.join(trial.path, model_path)

        model = osim.Model(model_path)

        # Preferred: OpenSim total mass
        try:
            state = model.initSystem()
            total_mass = model.getTotalMass(state)
        except Exception:
            # Fallback: sum body masses
            body_set = model.getBodySet()
            total_mass = sum(body_set.get(i).getMass() for i in range(body_set.getSize()))

        # total_mass = trial.get_body_mass_from_grf()

        model_weights[model_name] = total_mass
        print(f"{model_name}: {total_mass:.3f} kg")
    except Exception as e:
        print(f"Could not load mass for {model_name}: {e}")

# Plot bar graph
names = list(model_weights.keys())
masses = [model_weights[n] for n in names]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(names, masses, color=[colors.get(n, "gray") for n in names])

ax.set_title(f"Model Weights Comparison - {trialName}")
ax.set_ylabel("Total model mass (kg)")
ax.set_xticklabels(names, rotation=20, ha="right")

# Add value labels
for bar, m in zip(bars, masses):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f"{m:.2f}",
            ha='center', va='bottom', fontsize=9)

plt.tight_layout()

save_path = os.path.join(results_dir, f"model_weights_{trialName}.png")
plt.savefig(save_path, dpi=300)
print(f"Saved model weights plot: {save_path}")

## Compare models muscles optimal fibre length, isometric strength, MA in anatomical potions



In [ ]:

force_threshold = 300
models = {}
muscleList = set()
for model_name, trial in trials.items():
    if not isinstance(trial, utils.Analyse):
        continue
    models[model_name] = openSim.osim.Model(os.path.join(trial.path, trial.model_dir))
    for muscle in models[model_name].getMuscles():
        if muscle.getMaxIsometricForce() > force_threshold:
            muscleList.add(muscle.getName())

muscleList = sorted(list(muscleList))
model_names = list(models.keys())

def plot_muscle_param(param_name, getter_fn, ylabel, filename):
    """One subplot per muscle, bars = models."""
    n_muscles = len(muscleList)
    n_cols = 5
    n_rows = int(np.ceil(n_muscles / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3, n_rows * 3), sharey=False)
    axes = axes.flatten()

    bar_colors = [colors.get(m, 'steelblue') for m in model_names]

    for idx, muscle_name in enumerate(muscleList):
        ax = axes[idx]
        vals = []
        for mname in model_names:
            try:
                val = getter_fn(models[mname].getMuscles().get(muscle_name))
            except:
                val = 0
            vals.append(val)
        ax.bar(range(len(model_names)), vals, color=bar_colors)
        ax.set_title(muscle_name, fontsize=8)
        ax.set_xticks([])
        ax.tick_params(axis='y', labelsize=7)

    # Hide unused subplots
    for idx in range(n_muscles, len(axes)):
        axes[idx].set_visible(False)

    # Shared legend below
    handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in bar_colors]
    fig.legend(handles, model_names, loc='lower center', ncol=len(model_names),
               fontsize='x-small', bbox_to_anchor=(0.5, 0))
    fig.suptitle(f'{param_name} — {trialName}', fontsize=12)
    fig.text(0.01, 0.5, ylabel, va='center', rotation='vertical', fontsize=9)
    plt.tight_layout(rect=[0.03, 0.06, 1, 0.96])

    path = os.path.join(results_dir, f"{filename}_{trialName}.png")
    fig.savefig(path, dpi=150)
    print(f"Saved: {path}")
    plt.show()

plot_muscle_param(
    'Max Isometric Force',
    lambda m: m.getMaxIsometricForce(),
    'Force (N)',
    'muscle_max_iso_force'
)

plot_muscle_param(
    'Pennation Angle',
    lambda m: np.degrees(m.getPennationAngleAtOptimalFiberLength()),
    'Angle (deg)',
    'muscle_pennation_angle'
)

plot_muscle_param(
    'Optimal Fiber Length',
    lambda m: m.getOptimalFiberLength(),
    'Length (m)',
    'muscle_optimal_fiber_length'
)


In [ ]:
path = r"C:\Git\powerlifing_model_clean\models\Athlete_03_Lernagopal\25_03_31\scaled_89_increased_3.00.osim"
model = osim.Model(path)
state = model.initSystem()
total_mass = model.getTotalMass(state)
print(f"Total model mass: {total_mass:.3f} kg")

In [ ]:
# Compare model range of motion (ROM) across DOFs using loaded `trials` and `ik_columns`

try:
    plt
except NameError:
    import matplotlib.pyplot as plt

rom_rows = []

for model_name, trial_obj in trials.items():
    ik_path = os.path.join(trial_obj.path, trial_obj.ik)
    ik_df = utils.load_any_data_file(ik_path)

    for dof in ik_columns:
        if dof in ik_df.columns:
            rom_val = ik_df[dof].max() - ik_df[dof].min()
            rom_rows.append(
                {"model": model_name, "dof": dof, "rom": rom_val}
            )
        else:
            print(f"[WARN] DOF '{dof}' not found in IK file for model '{model_name}'")

rom_df = pd.DataFrame(rom_rows)

# Save table
rom_csv_path = os.path.join(results_dir, f"rom_comparison_{trialName}.csv")
rom_df.to_csv(rom_csv_path, index=False)
print(f"Saved ROM table: {rom_csv_path}")

# Plot grouped bar chart
pivot_rom = rom_df.pivot(index="dof", columns="model", values="rom")

ax = pivot_rom.plot(
    kind="bar",
    figsize=(14, 6),
    width=0.85,
    color=[colors.get(c, "gray") for c in pivot_rom.columns]
)
ax.set_title(f"Range of Motion Comparison - {trialName}")
ax.set_xlabel("DOF")
ax.set_ylabel("ROM (IK units)")
ax.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()

rom_plot_path = os.path.join(results_dir, f"rom_comparison_{trialName}.png")
plt.savefig(rom_plot_path, dpi=300)
print(f"Saved ROM plot: {rom_plot_path}")

rom_df

## Compare segmetn lengths

In [ ]:
# Compare segment lengths across models using marker locations from IK
# Segment lengths are estimated as Euclidean distances between adjacent markers

segment_marker_pairs = {
    'Thigh_R':  ('R.ASIS', 'R.Knee.Lat'),
    'Shank_R':  ('R.Knee.Lat', 'R.Ankle.Lat'),
    'Foot_R':   ('R.Ankle.Lat', 'R.Toe'),
    'Thigh_L':  ('L.ASIS', 'L.Knee.Lat'),
    'Shank_L':  ('L.Knee.Lat', 'L.Ankle.Lat'),
    'Foot_L':   ('L.Ankle.Lat', 'L.Toe'),
}

segment_lengths = {}

for model_name, trial_obj in trials.items():
    # Look for marker locations file
    marker_file = None
    for candidate in [
        os.path.join(trial_obj.path, 'ik_model_marker_locations.sto'),
        os.path.join(trial_obj.path, f'{trialName}_ik_model_marker_locations.sto'),
    ]:
        if os.path.exists(candidate):
            marker_file = candidate
            break

    if marker_file is None:
        # Try to find any *marker_locations.sto in the trial path
        matches = [f for f in os.listdir(trial_obj.path) if 'marker_locations' in f and f.endswith('.sto')]
        if matches:
            marker_file = os.path.join(trial_obj.path, matches[0])

    if marker_file is None:
        print(f"[WARN] No marker locations file found for '{model_name}', skipping.")
        continue

    markers_df = utils.load_any_data_file(marker_file)
    lengths = {}

    for seg_name, (m1, m2) in segment_marker_pairs.items():
        # Marker columns are expected as markerName_tx, _ty, _tz  OR  markerName_x, _y, _z
        found = False
        for suffix in [('_tx', '_ty', '_tz'), ('_x', '_y', '_z')]:
            cols1 = [f'{m1}{s}' for s in suffix]
            cols2 = [f'{m2}{s}' for s in suffix]
            if all(c in markers_df.columns for c in cols1 + cols2):
                p1 = markers_df[cols1].values
                p2 = markers_df[cols2].values
                dist = np.linalg.norm(p1 - p2, axis=1).mean()
                lengths[seg_name] = dist
                found = True
                break
        if not found:
            print(f"[WARN] Markers for segment '{seg_name}' not found in '{model_name}'")

    segment_lengths[model_name] = lengths
    print(f"{model_name}: {lengths}")

# --- Plot ---
all_segments = list(segment_marker_pairs.keys())
x = np.arange(len(all_segments))
bar_width = 0.8 / max(len(segment_lengths), 1)

fig, ax_seg = plt.subplots(figsize=(14, 6))

for i, (model_name, lengths) in enumerate(segment_lengths.items()):
    vals = [lengths.get(seg, np.nan) for seg in all_segments]
    ax_seg.bar(x + i * bar_width, vals,
               width=bar_width,
               label=model_name,
               color=colors.get(model_name, 'gray'),
               alpha=0.85)

ax_seg.set_xticks(x + bar_width * (len(segment_lengths) - 1) / 2)
ax_seg.set_xticklabels(all_segments, rotation=20, ha='right')
ax_seg.set_ylabel('Mean Segment Length (m)')
ax_seg.set_title(f'Segment Length Comparison — {trialName}')
ax_seg.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize='small')
plt.tight_layout()

save_path_seg = os.path.join(results_dir, f'segment_lengths_{trialName}.png')
fig.savefig(save_path_seg, dpi=300)
print(f"Saved segment lengths plot: {save_path_seg}")

# Save to CSV
seg_df = pd.DataFrame(segment_lengths).T
seg_df.index.name = 'Model'
seg_csv = os.path.join(results_dir, f'segment_lengths_{trialName}.csv')
seg_df.to_csv(seg_csv)
print(f"Saved segment lengths CSV: {seg_csv}")
seg_df

## IK & ID

In [ ]:
n_cols = len(ik_columns)
n_rows = 2
fig, ax = plt.subplots(nrows=int(n_rows), ncols=int(n_cols), figsize=(2*n_cols, 3*n_rows), sharex='col')
plt.suptitle(f'Comparison - {trialName}', y=0.98, fontsize=10)

models_to_flip = ['Lernagopal', 'Scaled (GPK)']  # Add model names that require flipping here
models_to_flip = ['Lernagopal', 'GPK']

for trial_name, trial in trials.items():
    angles = utils.load_any_data_file(os.path.join(trial.path, trial.ik))
    moments = utils.load_any_data_file(os.path.join(trial.path, trial.id)) 
    for col_idx, col_name in enumerate(ik_columns):
        
        if col_name == 'time':
            continue
        
        if any(model in trial_name for model in models_to_flip) and col_name.__contains__('knee_angle'):
            print(trial_name + '_ flipped') 
            flip_values = -1
        else:
            flip_values = 1
        
        try:
            ax[0, col_idx].plot(angles['time'], angles[col_name] * flip_values, label=trial_name, color=colors[trial_name], linestyle=lineStyles[trial_name])
            ax[1, col_idx].plot(moments['time'], moments[col_name+'_moment']* flip_values, label=trial_name, color=colors[trial_name], linestyle=lineStyles[trial_name])
        except KeyError:
            print(f"Column {col_name} not found in {trial_name} data.")
        
        ax[0, col_idx].set_title(f'{col_name}', fontsize=8)
        ax[1, col_idx].set_xlabel('Time (s)')
        if col_idx == 0:
            ax[0, col_idx].set_ylabel('Angle (deg)')
            ax[1, col_idx].set_ylabel('Moment (Nm)')           
        
# add legend outside of the plot        
white_right_margin = 0.2
handles, labels_legend = ax[0, 0].get_legend_handles_labels()
plt.tight_layout()
plt.subplots_adjust(right=1 - white_right_margin)
fig.legend(handles, labels_legend, loc='center left', bbox_to_anchor=(1 - white_right_margin + 0.02, 0.5))

save_path = os.path.join(results_dir, f'external_biomech_{trialName}.png')
plt.savefig(save_path)
print(f"Inverse kinematics comparison plot saved: {save_path}")

## Muscle Forces

In [ ]:
mf_data = {}
for label, trial in trials.items():
    mf_data[label] = {
        'so': utils.load_any_data_file(os.path.join(trial.path, trial.so_forces)),
        'ceinms': utils.load_any_data_file(os.path.join(trial.path, trial.jra_forces_ceinms))
    }

n_cols = 5
n_rows = int(np.ceil(len(muscleGroups) / n_cols))

fig, ax = plt.subplots(
    nrows=n_rows,
    ncols=n_cols,
    figsize=(4.8 * n_cols, 2.8 * n_rows),
    sharex=True
)
ax = ax.flatten()

for count, (muscleGroup, muscles) in enumerate(muscleGroups.items()):
    ax[count].set_title(muscleGroup, fontsize=10)

    for model in labels:
        so_df = mf_data[model]['so']
        ceinms_df = mf_data[model]['ceinms']

        try:
            ax[count].plot(
                so_df['time'],
                so_df[muscles].sum(axis=1),
                label=f'{model} SO',
                color=colors[model],
                linestyle=lineStyles[model],
                linewidth=1.5
            )
        except Exception:
            print(f"SO data for {model} not found for {muscleGroup}. Skipping SO plot.")

        try:
            ax[count].plot(
                ceinms_df['time'],
                ceinms_df[muscles].sum(axis=1),
                label=f'{model} CEINMS',
                color=colors[model],
                linestyle=lineStyles[model],
                linewidth=1.5
            )
        except Exception:
            print(f"CEINMS data for {model} not found for {muscleGroup}. Skipping CEINMS plot.")

# Hide unused subplots
for i in range(len(muscleGroups), len(ax)):
    ax[i].axis('off')

# Collect unique legend entries
handles, legend_labels = [], []
for a in ax[:len(muscleGroups)]:
    h, l = a.get_legend_handles_labels()
    for hh, ll in zip(h, l):
        if ll not in legend_labels:
            handles.append(hh)
            legend_labels.append(ll)

# Bottom legend (prevents big right-side whitespace)
fig.legend(
    handles,
    legend_labels,
    loc='lower center',
    bbox_to_anchor=(0.5, -0.01),
    ncol=4,
    frameon=False
)

# Tighter spacing between subplots
fig.subplots_adjust(left=0.05, right=0.98, top=0.95, bottom=0.10, wspace=0.18, hspace=0.30)

save_path = os.path.join(results_dir, f'muscle_forces_{trialName}.png')
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f"Muscle forces comparison plot saved: {save_path}")

## Muscle moments contributions to each moment

In [ ]:
def plot_muscle_moments(ax: plt.Axes, trial: utils.Analyse, dof: str, forces: str = 'so'):
    '''
    Plots muscle moments for a given degree of freedom (DOF) on the provided axes.
    Parameters:
        - ax: The matplotlib axes to plot on.
        - trial: The trial data containing paths to the necessary files.
        - dof: The degree of freedom for which to plot the muscle moments (e.g., 'hip_flexion_r').
        - forces: The type of muscle forces to use ('so' for static optimization or 'ceinms' for electromyography informed optimization).
    '''
    moments = utils.load_any_data_file(os.path.join(trial.path, trial.id))

    if forces.lower() == 'so':
        muscle_forces = utils.load_any_data_file(os.path.join(trial.path, trial.so_forces))
    elif forces.lower() == 'ceinms':
        muscle_forces = utils.load_any_data_file(os.path.join(trial.path, trial.jra_forces_ceinms))
    ma_path = os.path.join(trial.path, trial.ma, f'_MuscleAnalysis_MomentArm_{dof}.sto')
    if not os.path.exists(ma_path):
        print(f"Moment arm file for {dof} not found in {trial.path}. Skipping muscle moment plot for this DOF.")
        return
    
    try:
        moment_arms = utils.load_any_data_file(os.path.join(trial.path, trial.ma, f'_MuscleAnalysis_MomentArm_{dof}.sto'))
    except:
        print(f"Moment arm file for {dof} not found in {trial.path}. Skipping muscle moment plot for this DOF.")
        return

    muscle_list = muscle_forces.columns.drop('time')

    muscles = openSim.find_non_zero_mom_arm_muscles(moment_arms, muscle_list)
    # print(f"Non-zero moment arm muscles for {dof}: {muscles}")

    muscle_moments = muscle_forces.multiply(moment_arms, axis=0)
    muscle_moments['time'] = muscle_forces['time']


    for muscle in muscles:
        ax.plot(muscle_moments['time'], muscle_moments[muscle], label=muscle, linestyle='--')

    ax.plot(moments['time'], moments[dof+'_moment'], label=f'Inverse Dynamics {model_name}', color=colors[model_name], linewidth=2)

    # Fill area without edge styling
    ax.fill_between(
        muscle_moments['time'],
        muscle_moments[muscles].sum(axis=1),
        alpha=0.3,
        color='gray',
    )

    # Add dashed outline separately
    ax.plot(
        muscle_moments['time'],
        muscle_moments[muscles].sum(axis=1),
        color='black',
        linestyle='--',
        linewidth=2,
        label='Total Muscle Moment'
    )

    # calculate the difference between the total muscle moment and the inverse dynamics moment and add it as text to the plot
    total_muscle_moment = muscle_moments[muscles].sum(axis=1)
    
    if moments[dof+'_moment'] is not None:
        inverse_dynamics_moment = moments[dof+'_moment']
        
    elif moments[dof+'_force'] is not None:    
        inverse_dynamics_moment = moments[dof+'_force']
        
    moment_diff = total_muscle_moment - inverse_dynamics_moment
    moment_diff_mean = moment_diff.mean()
    moment_diff_std = moment_diff.std()

    moment_diff_mean_pct = (moment_diff_mean / (total_muscle_moment.max() - total_muscle_moment.min())) * 100
    moment_diff_std_pct = (moment_diff_std / (total_muscle_moment.max() - total_muscle_moment.min())) * 100

    text_str = f'Mean Residual: {moment_diff_mean:.2f} Nm ({moment_diff_mean_pct:.2f}%) \nStd: {moment_diff_std:.2f} Nm ({moment_diff_std_pct:.2f}%)'
    ax.text(0.02, 0.98, text_str, transform=ax.transAxes, fontsize=10, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))


fig, axes = plt.subplots(nrows=len(ik_columns), ncols=len(trials), figsize=(25, 15))

for irow, dof in enumerate(ik_columns):
    for icol, model_name in enumerate(trials.keys()):
        ax = axes[irow, icol]
        try:
            plot_muscle_moments(ax, trials[model_name], dof, forces=forces_type[model_name])
        except Exception as e:
            print(f"Error plotting muscle moments for {model_name}, {dof}: {e}")
            print(f'check folder {trials[model_name].path} for missing files.')

        if icol == 0:
            ax.set_ylabel(f'{dof} (Nm)')
        
        if irow == len(ik_columns) - 1:
            ax.set_xlabel('Time (s)')
        elif irow == 0:
            ax.set_title(f'{model_name}', fontsize=12)

        if icol == len(trials) - 1 and irow == 0:
            ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize='small')

# add sup title
fig.suptitle(f'Muscle Moments Comparison - {trialName}', fontsize=16, y=0.90)

# Sync y-axis limits per row
n_rows = len(ik_columns)
n_cols = len(trials)
for row_idx in range(n_rows):
    y_min = min(axes[row_idx, col_idx].get_ylim()[0] for col_idx in range(n_cols))
    y_max = max(axes[row_idx, col_idx].get_ylim()[1] for col_idx in range(n_cols))
    for col_idx in range(n_cols):
        axes[row_idx, col_idx].set_ylim(y_min, y_max)

# save figure
save_path = os.path.join(results_dir, f'muscle_moments_{trialName}.png')
fig.tight_layout(h_pad=2.0, w_pad=.5)
fig.subplots_adjust(hspace=0.5, wspace=0.3) 
fig.savefig(save_path, dpi=300)
print(f"Muscle moments comparison plot saved: {save_path}")


## Interactive

In [ ]:
save_path = save_path.replace('.png', '.html')
utils.convert_to_interactive_fig(fig, save_path, launch_browser=False)
print(f"Muscle moments comparison interactive plot saved: {save_path}")

## Summary errors IK, Moments, and EMG

In [ ]:
def markers_per_dof(osim_model: osim.Model, dof: str):
    """
    Return marker names relevant to a DOF using marker parent frame + side filtering.
    """
    dof_lower = dof.lower()

    # Side filter from DOF
    side = "_r" if "_r" in dof_lower else "_l" if "_l" in dof_lower else None

    # Joint/DOF -> likely body/frame keywords
    dof_to_frames = {
        "hip": ["","pelvis", "femur"],
        "knee": ["femur", "tibia", "patella"],
        "ankle": ["tibia", "talus", "calcn", "foot"],
        "subtalar": ["talus", "calcn", "foot"],
        "mtp": ["calcn", "toes", "foot"],
        "lumbar": ["pelvis", "torso", "lumbar"],
        "arm": ["clavicle", "scapula", "humerus", "torso"],
        "elbow": ["humerus", "ulna", "radius"],
        "pro_sup": ["ulna", "radius"],
        "wrist": ["ulna", "radius", "hand"],
    }

    # Detect DOF family
    dof_family = None
    for k in dof_to_frames:
        if k in dof_lower:
            dof_family = k
            break
    if dof_family is None:
        return []

    frame_keywords = dof_to_frames[dof_family]
    markerset = osim_model.getMarkerSet()
    relevant_markers = []

    for i in range(markerset.getSize()):
        marker = markerset.get(i)
        marker_name = marker.getName()

        # OpenSim API compatibility across versions
        try:
            parent_name = marker.getParentFrameName()
        except AttributeError:
            try:
                parent_name = marker.getParentFrame().getName()
            except Exception:
                parent_name = ""

        parent_name = str(parent_name).lower()

        # Frame relevance
        if not any(k in parent_name for k in frame_keywords):
            continue

        # Side relevance (match either marker name or parent frame)
        if side is not None:
            side_tokens = ["_r", "r.", "right"] if side == "_r" else ["_l", "l.", "left"]
            marker_name_l = marker_name.lower()
            if not (any(t in marker_name_l for t in side_tokens) or any(t in parent_name for t in side_tokens)):
                continue

        relevant_markers.append(marker_name)

    return sorted(set(relevant_markers))


# Comprehensive summary: IK marker errors, Moment R²/RMSE%, EMG R²/RMSE%
# Layout: rows = metrics (5), columns = DOFs + All DOFs aggregate, colors = models

import opensim as osim

# Invert EMG mapping: muscle -> EMG channel
emg_mapping = settings.EMG_muscle_mapping
muscle_to_emg = {}
for channel, muscle_list in emg_mapping.items():
    for m in muscle_list:
        muscle_to_emg[m] = channel

# Pre-cache markers_per_dof for each unique model (one osim.Model load per model)
_dof_markers_cache = {}  # model_name -> {dof -> [marker_names]}
for _lbl, _trial in trials.items():
    _model_full = os.path.join(_trial.path, _trial.model_dir)
    if not os.path.exists(_model_full):
        _dof_markers_cache[_lbl] = {}
        continue
    try:
        _osim_model = osim.Model(_model_full)
        _dof_markers_cache[_lbl] = {
            dof: markers_per_dof(_osim_model, dof) for dof in ik_columns
        }
    except Exception as _e:
        print(f"Could not load model for {_lbl}: {_e}")
        _dof_markers_cache[_lbl] = {}

summary_all = pd.DataFrame(columns=['DOF', 'Model', 'IK_Marker_Error', 'Moment_RMSE_pct', 'Moment_R2', 'EMG_RMSE_pct', 'EMG_R2'])

for model_name, trial in trials.items():
    # Load files — skip gracefully on missing
    _err_all_path = os.path.join(trial.path, '_ik_marker_errors_all.sto')
    try:
        ik_marker_errors = utils.load_any_data_file(_err_all_path) if os.path.exists(_err_all_path) else None
    except Exception:
        ik_marker_errors = None

    try:
        moments = utils.load_any_data_file(os.path.join(trial.path, trial.id))
    except Exception:
        moments = None

    force_type = forces_type[model_name]
    if force_type.upper() == 'SO':
        muscle_forces_path = os.path.join(trial.path, trial.so_forces)
        activations_path   = os.path.join(trial.path, trial.so_activations)
    else:
        muscle_forces_path = os.path.join(trial.path, trial.jra_forces_ceinms)
        activations_path   = os.path.join(trial.path, trial.jra_forces_ceinms.replace('MuscleForces.sto', 'AdjustedEmgs.sto'))

    muscle_forces = utils.load_any_data_file(muscle_forces_path) if os.path.exists(muscle_forces_path) else None
    activations   = utils.load_any_data_file(activations_path)   if os.path.exists(activations_path)   else None

    try:
        emg = utils.load_any_data_file(os.path.join(trial.path, trial.emg))
    except Exception:
        emg = None

    _dof_markers = _dof_markers_cache.get(model_name, {})

    for dof in ik_columns:
        row = {
            'DOF': dof, 'Model': model_name,
            'IK_Marker_Error': np.nan, 'Moment_RMSE_pct': np.nan, 'Moment_R2': np.nan,
            'EMG_RMSE_pct': np.nan, 'EMG_R2': np.nan
        }

        # --- Row 1: Mean marker error (mm) for markers relevant to this DOF ---
        if ik_marker_errors is not None:
            _dof_mkrs = _dof_markers.get(dof, [])
            # keep only markers that appear in the error file
            _available = [mk for mk in _dof_mkrs if mk in ik_marker_errors.columns]
            if _available:
                try:
                    # mean over time, then mean over markers → scalar in mm
                    row['IK_Marker_Error'] = ik_marker_errors[_available].mean(axis=1).mean() * 1000
                except Exception as e:
                    print(f"IK marker error for {model_name}, {dof}: {e}")
            else:
                # fallback: overall RMS error if no markers matched
                if 'marker_error_RMS' in ik_marker_errors.columns:
                    row['IK_Marker_Error'] = ik_marker_errors['marker_error_RMS'].mean() * 1000

        # --- Rows 2 & 3: Moment RMSE% and R² ---
        if moments is not None and muscle_forces is not None and dof + '_moment' in moments.columns:
            ma_path = os.path.join(trial.path, trial.ma, f'_MuscleAnalysis_MomentArm_{dof}.sto')
            if os.path.exists(ma_path):
                try:
                    moment_arms = utils.load_any_data_file(ma_path)
                    ml = muscle_forces.columns.drop('time')
                    muscles_dof = openSim.find_non_zero_mom_arm_muscles(moment_arms, ml)
                    if len(muscles_dof) > 0:
                        total_mm = muscle_forces.multiply(moment_arms, axis=0)[muscles_dof].sum(axis=1).values
                        id_m = moments[dof + '_moment'].values
                        n = min(len(id_m), len(total_mm))
                        id_m, total_mm = id_m[:n], total_mm[:n]
                        id_range = id_m.max() - id_m.min()
                        rmse_val = utils.rmse(id_m, total_mm)
                        row['Moment_R2']       = utils.rsquared(id_m, total_mm)
                        row['Moment_RMSE_pct'] = (rmse_val / id_range) * 100 if id_range != 0 else np.nan
                except Exception as e:
                    print(f"Moment error for {model_name}, {dof}: {e}")

        # --- Rows 4 & 5: EMG RMSE% and R² (activations vs measured EMG) ---
        if activations is not None and emg is not None and muscle_forces is not None:
            ma_path = os.path.join(trial.path, trial.ma, f'_MuscleAnalysis_MomentArm_{dof}.sto')
            if os.path.exists(ma_path):
                try:
                    moment_arms = utils.load_any_data_file(ma_path)
                    ml = muscle_forces.columns.drop('time')
                    muscles_dof = openSim.find_non_zero_mom_arm_muscles(moment_arms, ml)
                    act_vals, emg_vals = [], []
                    for m in muscles_dof:
                        if m not in activations.columns:
                            continue
                        emg_ch = muscle_to_emg.get(m)
                        if emg_ch is None or emg_ch not in emg.columns:
                            continue
                        n = min(len(activations[m]), len(emg[emg_ch]))
                        act_vals.append(activations[m].values[:n])
                        emg_vals.append(emg[emg_ch].values[:n])
                    if act_vals:
                        act_arr = np.concatenate(act_vals)
                        emg_arr = np.concatenate(emg_vals)
                        emg_range = emg_arr.max() - emg_arr.min()
                        rmse_emg = utils.rmse(emg_arr, act_arr)
                        row['EMG_R2']       = utils.rsquared(emg_arr, act_arr)
                        row['EMG_RMSE_pct'] = (rmse_emg / emg_range) * 100 if emg_range != 0 else np.nan
                except Exception as e:
                    print(f"EMG error for {model_name}, {dof}: {e}")

        summary_all = pd.concat([summary_all, pd.DataFrame([row])], ignore_index=True)

save_path = os.path.join(results_dir, f'summary_all_metrics_{trialName}.csv')
summary_all.to_csv(save_path, index=False)
print(f"Saved comprehensive summary metrics: {save_path}")

# ---- Plot ----
metric_rows = [
    ('IK_Marker_Error', 'Marker Error\n(mm)',  False),
    ('Moment_RMSE_pct', 'Moment\nRMSE (%)',    False),
    ('Moment_R2',       'Moment R²',           True),
    ('EMG_RMSE_pct',    'EMG\nRMSE (%)',       False),
    ('EMG_R2',          'EMG R²',              True),
]

n_metrics   = len(metric_rows)
n_dofs      = len(ik_columns)
n_cols_plot = n_dofs + 1  # +1 for aggregated All DOFs column

fig, axes = plt.subplots(nrows=n_metrics, ncols=n_cols_plot,
                         figsize=(3.2 * n_cols_plot, 3.0 * n_metrics))
fig.suptitle(f'Summary Metrics — {trialName}', fontsize=13, y=1.01)

x_pos = np.arange(len(labels))
bar_c = [colors.get(m, 'gray') for m in labels]

for row_idx, (metric, row_label, is_r2) in enumerate(metric_rows):

    # --- Per-DOF bar charts ---
    for col_idx, dof in enumerate(ik_columns):
        ax = axes[row_idx, col_idx]
        dof_df = summary_all[summary_all['DOF'] == dof]
        vals = [
            float(dof_df.loc[dof_df['Model'] == m, metric].values[0])
            if m in dof_df['Model'].values else np.nan
            for m in labels
        ]

        display_vals = [v if not np.isnan(v) else 0 for v in vals]
        bars = ax.bar(x_pos, display_vals, color=bar_c)
        # Hatch bars where data is missing so the model still appears in the row
        for bar, v in zip(bars, vals):
            if np.isnan(v):
                bar.set_hatch('//')
                bar.set_edgecolor('gray')
                bar.set_alpha(0.35)

        if is_r2:
            ax.axhline(1.0, color='black', linestyle='--', linewidth=0.8)

        if row_idx == 0:
            ax.set_title(dof, fontsize=8)
        if col_idx == 0:
            ax.set_ylabel(row_label, fontsize=8)
        ax.set_xticks([])

    # --- Aggregated All DOFs column (boxplot + jitter + mean) ---
    agg_ax = axes[row_idx, n_dofs]
    by_model = [summary_all.loc[summary_all['Model'] == m, metric].dropna().values for m in labels]

    bp = agg_ax.boxplot(by_model, positions=x_pos, patch_artist=True, widths=0.6, showfliers=False)
    for i, patch in enumerate(bp['boxes']):
        patch.set_facecolor(colors.get(labels[i], 'gray'))
        patch.set_alpha(0.5)
        if len(by_model[i]) == 0:
            patch.set_hatch('//')
            patch.set_edgecolor('gray')
    for i, vals in enumerate(by_model):
        if len(vals) > 0:
            jitter = np.random.normal(0, 0.05, len(vals))
            agg_ax.scatter(np.full(len(vals), x_pos[i]) + jitter, vals,
                           s=15, color=colors.get(labels[i], 'gray'), alpha=0.9, zorder=2)
            agg_ax.scatter(x_pos[i], np.mean(vals),
                           marker='D', s=30, color='black', zorder=3)

    if is_r2:
        agg_ax.axhline(1.0, color='black', linestyle='--', linewidth=0.8)
    agg_ax.set_xticks(x_pos)
    agg_ax.set_xticklabels([])
    if row_idx == 0:
        agg_ax.set_title('All DOFs', fontsize=8)

    # --- Sync y-limits across all columns in this row ---
    row_axes = [axes[row_idx, c] for c in range(n_cols_plot)]
    all_vals_row = summary_all[metric].dropna().values
    if len(all_vals_row):
        if is_r2:
            row_lo, row_hi = min(0, all_vals_row.min() * 0.95), 1.15
        else:
            row_lo = min(0, all_vals_row.min() - abs(all_vals_row.max()) * 0.05)
            row_hi = all_vals_row.max() * 1.15
        for _ax in row_axes:
            _ax.set_ylim([row_lo, row_hi])

# Legend below figure
handles_leg = [plt.Rectangle((0, 0), 1, 1, color=colors.get(m, 'gray')) for m in labels]
fig.legend(handles_leg, labels, loc='lower center', ncol=4, fontsize='small', bbox_to_anchor=(0.5, -0.01))

plt.tight_layout(rect=[0, 0.04, 1, 1])
save_path = os.path.join(results_dir, f'summary_all_metrics_{trialName}.png')
fig.savefig(save_path, dpi=300, bbox_inches='tight')
print(f"Summary plot saved: {save_path}")

summary_all


In [ ]:

# IK Marker Errors — mean ± SD of marker_error_RMS across all available trials
# Deduplicated by unique subject (SO and CEINMS share the same IK solution)

import glob

ik_err_col = 'marker_error_RMS'

# One entry per UNIQUE subject (first label that maps to that subject)
seen_subjects: dict = {}
unique_model_labels: list = []
for _lbl in labels:
    _subj = subject_names[_lbl]
    if _subj not in seen_subjects:
        seen_subjects[_subj] = _lbl
        unique_model_labels.append(_lbl)

# Collect mean ± SD per unique model per trial
ik_err_data: dict = {}   # unique_label -> {trial_name -> (mean_mm, std_mm)}
for _lbl in unique_model_labels:
    _subj = subject_names[_lbl]
    _subj_dir = os.path.join(utils.SIMULATIONS_DIR, _subj, session)
    ik_err_data[_lbl] = {}
    for _trial_dir in sorted(glob.glob(os.path.join(_subj_dir, '*'))):
        _tn = os.path.basename(_trial_dir)
        # skip calibration / static folders
        if any(_tn.lower().startswith(p) for p in ('calibration', 'static')):
            continue
        _err_file = os.path.join(_trial_dir, '_ik_marker_errors.sto')
        if not os.path.exists(_err_file):
            continue
        try:
            _df = utils.load_any_data_file(_err_file)
            if ik_err_col in _df.columns:
                _vals = _df[ik_err_col].values * 1000  # m → mm
                ik_err_data[_lbl][_tn] = (_vals.mean(), _vals.std())
        except Exception as _e:
            print(f"Could not load IK errors for {_lbl}, {_tn}: {_e}")

# Union of trial names across all unique models (sorted)
all_ik_trials = sorted({t for d in ik_err_data.values() for t in d})
n_tr = len(all_ik_trials)
n_um = len(unique_model_labels)

# --- Plot ---
fig_ik, ax_ik = plt.subplots(figsize=(max(8, n_tr * 1.8 + 2), 5))

_bar_w = 0.8 / n_um
_x     = np.arange(n_tr)

for _mi, _lbl in enumerate(unique_model_labels):
    _means = [ik_err_data[_lbl].get(t, (np.nan, np.nan))[0] for t in all_ik_trials]
    _stds  = [ik_err_data[_lbl].get(t, (np.nan, 0.0))[1]   for t in all_ik_trials]
    _offset = (_mi - n_um / 2 + 0.5) * _bar_w
    ax_ik.bar(_x + _offset, _means, width=_bar_w,
              color=colors.get(_lbl, 'gray'), label=_lbl, alpha=0.85)
    ax_ik.errorbar(_x + _offset, _means, yerr=_stds,
                   fmt='none', ecolor='black', capsize=3, linewidth=1.2)

ax_ik.set_xticks(_x)
ax_ik.set_xticklabels(all_ik_trials, rotation=30, ha='right', fontsize=9)
ax_ik.set_ylabel('Marker RMS Error (mm)')
ax_ik.set_title(f'IK Marker Errors — Mean ± SD across trial\n(session: {session})')
ax_ik.legend(loc='upper left', fontsize='small', ncol=1)

plt.tight_layout()
_save_ik = os.path.join(results_dir, 'ik_marker_errors_all_trials.png')
fig_ik.savefig(_save_ik, dpi=300, bbox_inches='tight')
print(f"IK marker errors plot saved: {_save_ik}")
plt.show()


In [ ]:

# Per-muscle EMG metrics: R², RME% (relative mean error / signed bias), RMSE%
# activation source: SO activations (SO models) or AdjustedEmgs (CEINMS models)

_muscle_emg_rows = []

for _model_name, _trial in trials.items():
    _force_type = forces_type[_model_name]
    if _force_type.upper() == 'SO':
        _act_path = os.path.join(_trial.path, _trial.so_activations)
    else:
        _act_path = os.path.join(_trial.path,
                                 _trial.jra_forces_ceinms.replace('MuscleForces.sto', 'AdjustedEmgs.sto'))

    if not os.path.exists(_act_path):
        print(f"Activations not found for {_model_name}: {_act_path}")
        continue

    try:
        _act  = utils.load_any_data_file(_act_path)
        _emg  = utils.load_any_data_file(os.path.join(_trial.path, _trial.emg))
    except Exception as _e:
        print(f"Could not load data for {_model_name}: {_e}")
        continue

    for _muscle in _act.columns:
        if _muscle == 'time':
            continue
        _emg_ch = muscle_to_emg.get(_muscle)
        if _emg_ch is None or _emg_ch not in _emg.columns:
            continue

        _n = min(len(_act[_muscle]), len(_emg[_emg_ch]))
        _a = _act[_muscle].values[:_n]
        _e = _emg[_emg_ch].values[:_n]
        _rng = _e.max() - _e.min()
        if _rng == 0:
            continue

        try:
            _r2       = utils.rsquared(_e, _a)
            _rmse_abs = utils.rmse(_e, _a)
            _rmse_pct = (_rmse_abs / _rng) * 100
            _rme_pct  = (np.mean(_a - _e) / _rng) * 100   # signed bias (+ = over-estimation)
            _muscle_emg_rows.append({
                'Model':       _model_name,
                'Muscle':      _muscle,
                'EMG_Channel': _emg_ch,
                'R2':          _r2,
                'RMSE_pct':    _rmse_pct,
                'RME_pct':     _rme_pct,
            })
        except Exception:
            pass

muscle_emg_df = pd.DataFrame(_muscle_emg_rows)

# Save CSV
_csv_path = os.path.join(results_dir, f'muscle_emg_metrics_{trialName}.csv')
muscle_emg_df.to_csv(_csv_path, index=False)
print(f"Per-muscle EMG metrics saved: {_csv_path}")

# ---- Heatmaps: rows = muscles, columns = models --------------------------------
_metrics_to_plot = [
    ('R2',       'R²',          0.0, 1.0,   'RdYlGn'),
    ('RMSE_pct', 'RMSE (%)',    0.0, None,  'YlOrRd'),
    ('RME_pct',  'RME (%)\n(+ = over)', None, None, 'RdBu_r'),
]

_all_muscles = sorted(muscle_emg_df['Muscle'].unique())
_n_muscles   = len(_all_muscles)
_n_models    = len(labels)

if _n_muscles == 0:
    print("No muscle-EMG pairs found — check muscle_to_emg mapping and activations file columns.")
else:
    _fig_hm, _axes_hm = plt.subplots(
        nrows=1, ncols=3,
        figsize=(6 * 3, max(6, _n_muscles * 0.35 + 2)),
        constrained_layout=True
    )

    for _ax, (_col, _title, _vmin, _vmax, _cmap) in zip(_axes_hm, _metrics_to_plot):
        # Build matrix: rows = muscles, columns = models
        _mat = np.full((_n_muscles, _n_models), np.nan)
        for _ci, _mdl in enumerate(labels):
            _sub = muscle_emg_df[muscle_emg_df['Model'] == _mdl]
            for _ri, _mus in enumerate(_all_muscles):
                _row_val = _sub[_sub['Muscle'] == _mus][_col]
                if len(_row_val):
                    _mat[_ri, _ci] = float(_row_val.values[0])

        # Normalise colour limits if not fixed
        _finite = _mat[np.isfinite(_mat)]
        _v_min  = _vmin if _vmin is not None else (_finite.min() if len(_finite) else 0)
        _v_max  = _vmax if _vmax is not None else (_finite.max() if len(_finite) else 1)
        # For RME_pct make symmetric around 0
        if _col == 'RME_pct' and len(_finite):
            _abs_max = np.abs(_finite).max()
            _v_min, _v_max = -_abs_max, _abs_max

        _im = _ax.imshow(_mat, aspect='auto', cmap=_cmap, vmin=_v_min, vmax=_v_max)
        plt.colorbar(_im, ax=_ax, shrink=0.6)

        _ax.set_xticks(range(_n_models))
        _ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
        _ax.set_yticks(range(_n_muscles))
        _ax.set_yticklabels(_all_muscles, fontsize=7)
        _ax.set_title(_title, fontsize=10)

        # Annotate cells with numeric values
        for _ri in range(_n_muscles):
            for _ci in range(_n_models):
                _v = _mat[_ri, _ci]
                if np.isfinite(_v):
                    _ax.text(_ci, _ri, f'{_v:.2f}', ha='center', va='center',
                             fontsize=5, color='black')

    _fig_hm.suptitle(
        f'Per-muscle EMG Metrics — {trialName}',
        fontsize=12
    )

    _save_hm = os.path.join(results_dir, f'muscle_emg_metrics_heatmap_{trialName}.png')
    _fig_hm.savefig(_save_hm, dpi=200, bbox_inches='tight')
    print(f"Heatmap saved: {_save_hm}")
    plt.show()

muscle_emg_df


## Moment arms

In [ ]:
def plot_moment_arms(ax: plt.Axes, trial: utils.Analyse, dof: str):
    '''
    Plots muscle moment arms for a given degree of freedom (DOF) on the provided axes.
    Parameters:
        - ax: The matplotlib axes to plot on.
        - trial: The trial data containing paths to the necessary files.
        - dof: The degree of freedom for which to plot the muscle moment arms (e.g., 'hip_flexion_r').
    '''

    moment_arms_path = os.path.join(trial.path, trial.ma, f'_MuscleAnalysis_MomentArm_{dof}.sto')
    if not os.path.exists(moment_arms_path):
        print(f"Moment arm file for {dof} not found in {trial.path}. Skipping muscle moment plot for this DOF.")
        return
    
    moment_arms = utils.load_any_data_file(moment_arms_path)

    if moment_arms is None:
        print(f"Moment arm data for {dof} not found in {trial.path}. Skipping muscle moment plot for this DOF.")
        return

    muscle_list = moment_arms.columns.drop('time')

    muscles = openSim.find_non_zero_mom_arm_muscles(moment_arms, muscle_list)
    # print(f"Non-zero moment arm muscles for {dof}: {muscles}")

    for muscle in muscles:
        ax.plot(moment_arms['time'], moment_arms[muscle], label=muscle, linestyle='--')

    ax.plot(moment_arms['time'], moment_arms[muscles].mean(axis=1), label=f'Mean', color=colors[model_name], linewidth=2)



fig, axes = plt.subplots(nrows=len(ik_columns), ncols=len(trials), figsize=(25, 15))
plt.suptitle(f'Muscle Moments Comparison - {trialName}', fontsize=16, y=1.02)

for irow, dof in enumerate(ik_columns):
    for icol, model_name in enumerate(trials.keys()):
        ax = axes[irow, icol]
        trial = trials[model_name]
        plot_moment_arms(ax, trial, dof)

        if icol == 0:
            ax.set_ylabel(f'{dof} (Nm)')
        
        if irow == len(ik_columns) - 1:
            ax.set_xlabel('Time (s)')
            
        elif irow == 0:
            ax.set_title(f'{model_name}', fontsize=12)

        if icol == len(trials) - 1 and irow == 0:
            ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize='small')

# add sup title
fig.suptitle(f'Muscle Moments Comparison - {trialName}', fontsize=16, y=1.02)
fig.subplots_adjust(hspace=0.3, wspace=0.3)

# Sync y-axis limits per row
n_rows = len(ik_columns)
n_cols = len(trials)
for row_idx in range(n_rows):
    # Keep only subplots that actually contain plotted data
    row_axes_with_data = [
        axes[row_idx, col_idx]
        for col_idx in range(n_cols)
        if len(axes[row_idx, col_idx].lines) > 0
    ]

    # Skip row if no subplot has data
    if not row_axes_with_data:
        continue

    # Compute shared y-limits using only non-empty axes
    y_min = min(ax_i.get_ylim()[0] for ax_i in row_axes_with_data)
    y_max = max(ax_i.get_ylim()[1] for ax_i in row_axes_with_data)

    # Apply limits only to axes that have data
    for col_idx in range(n_cols):
        axes[row_idx, col_idx].set_ylim(y_min, y_max)

# save figure
save_path = os.path.join(results_dir, f'ma_{trialName}.png')

fig.savefig(save_path, dpi=300)
print(f"Muscle moments comparison plot saved: {save_path}")

## Interactive

In [ ]:
save_path = save_path.replace('.png', '.html')
utils.convert_to_interactive_fig(fig, save_path)
print(f"Muscle moments comparison interactive plot saved: {save_path}")

## Maximum moment plot

In [ ]:
def plot_maximum_moment(ax: plt.Axes, trial: utils.Analyse, dof: str):
    '''
    Plots muscle moment arms for a given degree of freedom (DOF) on the provided axes.
    Parameters:
        - ax: The matplotlib axes to plot on.
        - trial: The trial data containing paths to the necessary files.
        - dof: The degree of freedom for which to plot the muscle moment arms (e.g., 'hip_flexion_r').
    '''
    moments = utils.load_any_data_file(os.path.join(trial.path, trial.id))
    norm_fibre_lengths = utils.load_any_data_file(os.path.join(trial.path, trial.ma, f'_MuscleAnalysis_NormalizedFiberLength.sto'))
    
    moment_arms_path = os.path.join(trial.path, trial.ma, f'_MuscleAnalysis_MomentArm_{dof}.sto')
    if not os.path.exists(moment_arms_path):
        print(f"Moment arm file for {dof} not found in {trial.path}. Skipping muscle moment plot for this DOF.")
        return
    
    moment_arms = utils.load_any_data_file(moment_arms_path)

    if moment_arms is None:
        print(f"Moment arm data for {dof} not found in {trial.path}. Skipping muscle moment plot for this DOF.")
        return

    muscle_list = moment_arms.columns.drop('time')

    muscles = openSim.find_non_zero_mom_arm_muscles(moment_arms, muscle_list)
    # print(f"Non-zero moment arm muscles for {dof}: {muscles}")

    model = osim.Model(os.path.join(trial.path, trial.model_dir))
    state = model.initSystem()

    for muscle in muscles:
        max_isometric_force = model.getMuscles().get(muscle).getMaxIsometricForce()
        pennation_angle = model.getMuscles().get(muscle).getPennationAngle(state)
        activation = 1.0  # Assuming full activation for maximum moment calculation

        muscle_moment = activation * max_isometric_force * np.cos(pennation_angle) * moment_arms[muscle] * norm_fibre_lengths[muscle]
        
        ax.plot(moment_arms['time'], muscle_moment, label=muscle, linestyle='--')

    ax.plot(moment_arms['time'], moment_arms[muscles].mean(axis=1), label=f'Mean', color=colors[model_name], linewidth=2)



fig, axes = plt.subplots(nrows=len(ik_columns), ncols=len(trials), figsize=(25, 15))
plt.suptitle(f'Muscle Moments Comparison - {trialName}', fontsize=16, y=1.02)

for irow, dof in enumerate(ik_columns):
    for icol, model_name in enumerate(trials.keys()):
        ax = axes[irow, icol]
        trial = trials[model_name]
        plot_maximum_moment(ax, trial, dof)

        if icol == 0:
            ax.set_ylabel(f'{dof} (Nm)')
        
        if irow == len(ik_columns) - 1:
            ax.set_xlabel('Time (s)')
            
        elif irow == 0:
            ax.set_title(f'{model_name}', fontsize=12)

        if icol == len(trials) - 1 and irow == 0:
            ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize='small')

# add sup title
fig.suptitle(f'Muscle Moments Comparison - {trialName}', fontsize=16, y=1.02)
fig.subplots_adjust(hspace=0.3, wspace=0.3)

# Sync y-axis limits per row
n_rows = len(ik_columns)
n_cols = len(trials)
for row_idx in range(n_rows):
    # Keep only subplots that actually contain plotted data
    row_axes_with_data = [
        axes[row_idx, col_idx]
        for col_idx in range(n_cols)
        if len(axes[row_idx, col_idx].lines) > 0
    ]

    # Skip row if no subplot has data
    if not row_axes_with_data:
        continue

    # Compute shared y-limits using only non-empty axes
    y_min = min(ax_i.get_ylim()[0] for ax_i in row_axes_with_data)
    y_max = max(ax_i.get_ylim()[1] for ax_i in row_axes_with_data)

    # Apply limits only to axes that have data
    for col_idx in range(n_cols):
        axes[row_idx, col_idx].set_ylim(y_min, y_max)

# save figure
save_path = os.path.join(results_dir, f'max_moments_{trialName}.png')

fig.savefig(save_path, dpi=300)
print(f"Muscle moments comparison plot saved: {save_path}")

## Interactive

In [ ]:
save_path = save_path.replace('.png', '.html')
utils.convert_to_interactive_fig(fig, save_path, launch_browser=True)
print(f"Muscle moments comparison interactive plot saved: {save_path}")

## muscle activations so ceinms emg

In [ ]:
emg_mapping = settings.EMG_muscle_mapping

for muscle in emg_mapping.keys():
    print(f"{muscle} -> {emg_mapping[muscle]}")

## Interactive

## JRA

In [ ]:
def get_jra_columns(model_type='Cateli'):

    components = {}
    if model_type.__contains__('Cateli') or model_type.__contains__('GPK'):
        components['hip'] = ['hip_r_on_femur_r_in_femur_r_fx', 'hip_r_on_femur_r_in_femur_r_fy', 'hip_r_on_femur_r_in_femur_r_fz']
        components['knee'] = ['walker_knee_r_on_tibia_r_in_tibia_r_fx', 'walker_knee_r_on_tibia_r_in_tibia_r_fy', 'walker_knee_r_on_tibia_r_in_tibia_r_fz']
        components['ankle'] = ['ankle_r_on_talus_r_in_talus_r_fx', 'ankle_r_on_talus_r_in_talus_r_fy', 'ankle_r_on_talus_r_in_talus_r_fz']
    elif model_type.__contains__('Lernagopal') :
        components['hip'] = ['hip_r_on_femur_r_in_femur_r_fx', 'hip_r_on_femur_r_in_femur_r_fy', 'hip_r_on_femur_r_in_femur_r_fz']
        components['knee'] = ['Lerner_knee_r_on_sagittal_articulation_frame_r_in_sagittal_articulation_frame_r_fx', 'Lerner_knee_r_on_sagittal_articulation_frame_r_in_sagittal_articulation_frame_r_fy', 'Lerner_knee_r_on_sagittal_articulation_frame_r_in_sagittal_articulation_frame_r_fz']
        components['ankle'] = ['ankle_r_on_talus_r_in_talus_r_fx', 'ankle_r_on_talus_r_in_talus_r_fy', 'ankle_r_on_talus_r_in_talus_r_fz']

    return components

joint_row = {'hip': 0, 'knee': 1, 'ankle': 2}

fig, ax = plt.subplots(3, 4, figsize=(15, 8))

for icol, model_name in enumerate(trials.keys()):
    joint_loads = utils.load_any_data_file(os.path.join(trials[model_name].path, trials[model_name].jra))
    components = get_jra_columns(model_type=model_name)
    for joint, cols in components.items():
        # Check if all required columns exist in the data
        missing_cols = [c for c in cols if c not in joint_loads.columns]
        if missing_cols:
            print(f"Skipping {joint} for {model_name}: missing columns {missing_cols}")
            print(f"Available knee-related columns: {[c for c in joint_loads.columns if 'knee' in c.lower()]}")
            continue

        irow = joint_row[joint]
        x = joint_loads[cols[0]]
        y = joint_loads[cols[1]]
        z = joint_loads[cols[2]]
        resultant = np.linalg.norm([x, y, z], axis=0)
        ax[irow, 0].plot(joint_loads['time'], x, label=model_name, color=colors[model_name], linestyle=lineStyles[model_name])
        ax[irow, 1].plot(joint_loads['time'], y, label=model_name, color=colors[model_name], linestyle=lineStyles[model_name])
        ax[irow, 2].plot(joint_loads['time'], z, label=model_name, color=colors[model_name], linestyle=lineStyles[model_name])
        ax[irow, 3].plot(joint_loads['time'], resultant, label=model_name, color=colors[model_name], linestyle=lineStyles[model_name])

        ax[irow, 0].set_ylabel(f'{joint.capitalize()} Force (N)')

ax[0, 0].set_title('X (+ anterior, - posterior)')
ax[0, 1].set_title('Y (+ medial, - lateral)')
ax[0, 2].set_title('Z (+ superior, - inferior)')
ax[0, 3].set_title('Resultant Force Magnitude')

for col in range(4):
    ax[2, col].set_xlabel('Time (s)')

# Unique labels only
handles, labels = ax[0, 0].get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax[0, 0].legend(by_label.values(), by_label.keys())

plt.tight_layout()
save_path = os.path.join(results_dir, f'jra_{trialName}.png')
plt.savefig(save_path)
print(f"JRA comparison plot saved: {save_path}")


In [ ]:
import numpy as np

def get_jra_columns(model_type='Cateli'):

    components = {}
    if model_type.__contains__('Cateli') or model_type.__contains__('GPK'):
        components['hip'] = ['hip_r_on_femur_r_in_femur_r_fx', 'hip_r_on_femur_r_in_femur_r_fy', 'hip_r_on_femur_r_in_femur_r_fz']
        components['knee'] = ['walker_knee_r_on_tibia_r_in_tibia_r_fx', 'walker_knee_r_on_tibia_r_in_tibia_r_fy', 'walker_knee_r_on_tibia_r_in_tibia_r_fz']
        components['ankle'] = ['ankle_r_on_talus_r_in_talus_r_fx', 'ankle_r_on_talus_r_in_talus_r_fy', 'ankle_r_on_talus_r_in_talus_r_fz']
    elif model_type.__contains__('Lernagopal') :
        components['hip'] = ['hip_r_on_femur_r_in_femur_r_fx', 'hip_r_on_femur_r_in_femur_r_fy', 'hip_r_on_femur_r_in_femur_r_fz']
        components['knee'] = ['Lerner_knee_r_on_sagittal_articulation_frame_r_in_sagittal_articulation_frame_r_fx', 'Lerner_knee_r_on_sagittal_articulation_frame_r_in_sagittal_articulation_frame_r_fy', 'Lerner_knee_r_on_sagittal_articulation_frame_r_in_sagittal_articulation_frame_r_fz']
        components['ankle'] = ['ankle_r_on_talus_r_in_talus_r_fx', 'ankle_r_on_talus_r_in_talus_r_fy', 'ankle_r_on_talus_r_in_talus_r_fz']


    return components

fig, ax = plt.subplots(3, 4, figsize=(15, 8))

for icol, model_name in enumerate(trials.keys()):
    joint_loads = utils.load_any_data_file(os.path.join(trials[model_name].path, trials[model_name].jra))
    components = get_jra_columns(model_type=model_name)
    for joint, cols in components.items():
        # Check if all required columns exist in the data
        missing_cols = [c for c in cols if c not in joint_loads.columns]
        if missing_cols:
            print(f"Skipping {joint} for {model_name}: missing columns {missing_cols}")
            print(f"Available knee-related columns: {[c for c in joint_loads.columns if 'knee' in c.lower()]}")
            continue

        x = joint_loads[cols[0]]
        y = joint_loads[cols[1]]
        z = joint_loads[cols[2]]
        sum = np.linalg.norm([x, y, z], axis=0)
        ax[0, 0].plot(joint_loads['time'], x, label=model_name, color=colors[model_name], linestyle=lineStyles[model_name])
        ax[0, 1].plot(joint_loads['time'], y, label=model_name, color=colors[model_name], linestyle=lineStyles[model_name])
        ax[0, 2].plot(joint_loads['time'], z, label=model_name, color=colors[model_name], linestyle=lineStyles[model_name])
        ax[0, 3].plot(joint_loads['time'], sum, label=model_name, color=colors[model_name], linestyle=lineStyles[model_name])
    
        ax[0, 0].set_ylabel(f'{joint.capitalize()} Force (N)')
        ax[0, 0].set_title('X (+ anterior, - posterior)')
        ax[0, 1].set_title('Y (+ medial, - lateral)')
        ax[0, 2].set_title('Z (+ superior, - inferior)')
        ax[0, 3].set_title('Resultant Force Magnitude')
    
        ax[2, 0].set_xlabel('Time (s)')
        ax[2, 1].set_xlabel('Time (s)')
        ax[2, 2].set_xlabel('Time (s)')
        ax[2, 3].set_xlabel('Time (s)')

ax[0, 0].legend()
plt.tight_layout()
save_path = os.path.join(results_dir, f'jra_{trialName}.png')
plt.savefig(save_path)
print(f"JRA comparison plot saved: {save_path}")